# partioned_regression

> 对应代码：`EconometricsCode/code_in_notes/Chap.1/partioned_regression.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.1`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.1")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们先建立两个数据框（frame）：当前框装入家庭户数据 chfs2017_hh.dta，新建的 master 框装入包含户级标识 hhid 与农村虚拟变量 rural 的主表，并按 hhid 去重，保证接下来的一对一链接不会错位。

In [2]:
%%stata
clear all
use ../datasets/chfs2017_hh.dta
frame create master
frame master: use ../datasets/chfs2017_master.dta
frame master: keep hhid rural
frame master: duplicates drop hhid rural, force


. clear all

. use ../datasets/chfs2017_hh.dta

. frame create master

. frame master: use ../datasets/chfs2017_master.dta

. frame master: keep hhid rural

. frame master: duplicates drop hhid rural, force

Duplicates in terms of hhid rural

(87,001 observations deleted)

. 


frlink 以 hhid 为键在两个框之间建立一对一链接，frget 把 master 框中的 rural 变量取回当前框，相当于跨文件合并（merge）的轻量替代。

In [3]:
%%stata
frlink 1:1 hhid, frame(master)
frget rural, from(master)


. frlink 1:1 hhid, frame(master)
(all observations in frame default matched)

. frget rural, from(master)
(1 variable copied from linked frame)

. 


生成消费、收入与资产的对数，把非线性关系化为对数线性形式，也减轻右偏的影响。

In [4]:
%%stata
// 计算对数
gen log_comsump = log(total_consump)
gen log_income = log(total_income)
gen log_asset = log(b2003a)


. // 计算对数
. gen log_comsump = log(total_consump)

. gen log_income = log(total_income)
(827 missing values generated)

. gen log_asset = log(b2003a)
(39,372 missing values generated)

. 


先做完整的多元回归：以对数收入、对数资产和农村虚拟变量解释对数消费，并把对数收入的系数存入 local，作为稍后对照的基准。

In [5]:
%%stata
// 回归
reg log_comsump log_income log_asset rural
local coef=_b[log_income]


. // 回归
. reg log_comsump log_income log_asset rural

      Source |       SS           df       MS      Number of obs   =       638
-------------+----------------------------------   F(3, 634)       =     76.86
       Model |  118.386699         3  39.4622331   Prob > F        =    0.0000
    Residual |  325.495238       634  .513399428   R-squared       =    0.2667
-------------+----------------------------------   Adj R-squared   =    0.2632
       Total |  443.881937       637  .696831926   Root MSE        =    .71652

------------------------------------------------------------------------------
 log_comsump | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
  log_income |   .2088404   .0238195     8.77   0.000     .1620657    .2556151
   log_asset |   .0923692   .0167727     5.51   0.000     .0594324     .125306
       rural |  -.2996816   .0706476    -4.24   0.000    -.4384131     -.16

接下来验证Frisch–Waugh–Lovell定理：先把 log_income 对其余解释变量回归并取残差，这相当于把 log_income 中与 log_asset、rural 线性相关的部分"排除"掉。

In [6]:
%%stata
// 为了获得log_income的系数，如果分块回归：
reg log_income log_asset rural
predict resid_log_income, resid


. // 为了获得log_income的系数，如果分块回归：
. reg log_income log_asset rural

      Source |       SS           df       MS      Number of obs   =       638
-------------+----------------------------------   F(2, 635)       =     92.81
       Model |     264.506         2     132.253   Prob > F        =    0.0000
    Residual |  904.876022       635  1.42500161   R-squared       =    0.2262
-------------+----------------------------------   Adj R-squared   =    0.2238
       Total |  1169.38202       637  1.83576456   Root MSE        =    1.1937

------------------------------------------------------------------------------
  log_income | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
   log_asset |   .3314514   .0246544    13.44   0.000     .2830373    .3798654
       rural |  -.0504724   .1176832    -0.43   0.668    -.2815677    .1806229
       _cons |   7.853439    .306354    25.64   0.000      7.251

同样地，把被解释变量 log_comsump 也对 log_asset 与 rural 回归取残差，得到剔除控制变量影响后的消费部分。

In [7]:
%%stata
reg log_comsump log_asset rural
predict resid_log_comsump, resid


. reg log_comsump log_asset rural

      Source |       SS           df       MS      Number of obs   =       639
-------------+----------------------------------   F(2, 636)       =     69.64
       Model |  80.9138893         2  40.4569447   Prob > F        =    0.0000
    Residual |  369.486977       636  .580954367   R-squared       =    0.1796
-------------+----------------------------------   Adj R-squared   =    0.1771
       Total |  450.400867       638  .705957471   Root MSE        =     .7622

------------------------------------------------------------------------------
 log_comsump | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
   log_asset |   .1644241   .0157091    10.47   0.000     .1335761    .1952722
       rural |  -.3042582   .0751108    -4.05   0.000    -.4517533   -.1567631
       _cons |   9.495388   .1951115    48.67   0.000     9.112247    9.878529
---------------

用两组残差做无截距回归，所得的 resid_log_income 系数应当与多元回归中 log_income 的系数完全一致——这正是分块回归（partioned regression）的结论：多元回归中某变量的系数，度量的是该变量"剔除其他解释变量后"的独立解释力。

In [8]:
%%stata
reg resid_log_comsump resid_log_income, noconstant
local coef_partioned=_b[resid_log_income]
di "OLS系数=`coef'，分块回归系数=`coef_partioned'"


. reg resid_log_comsump resid_log_income, noconstant

      Source |       SS           df       MS      Number of obs   =       638
-------------+----------------------------------   F(1, 637)       =     77.23
       Model |  39.4655394         1  39.4655394   Prob > F        =    0.0000
    Residual |  325.522965       637  .511025063   R-squared       =    0.1081
-------------+----------------------------------   Adj R-squared   =    0.1067
       Total |  364.988504       638  .572082295   Root MSE        =    .71486

------------------------------------------------------------------------------
resid_log_~p | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
resid_log_~e |   .2088404   .0237644     8.79   0.000     .1621744    .2555064
------------------------------------------------------------------------------

. local coef_partioned=_b[resid_log_income]

. di "OLS系数=`coef'，分块回归系数=`co

最后比较两种相关：原始变量的简单相关系数混杂了控制变量的影响，而残差之间的相关系数即偏相关系数，反映控制其余变量后收入与消费的净相关。

In [9]:
%%stata
corr log_comsump log_income
corr resid_log_comsump resid_log_income


. corr log_comsump log_income
(obs=39,184)

             | log_co~p log_in~e
-------------+------------------
 log_comsump |   1.0000
  log_income |   0.5224   1.0000


. corr resid_log_comsump resid_log_income
(obs=638)

             | resid_~p resid_~e
-------------+------------------
resid_log_~p |   1.0000
resid_log_~e |   0.3288   1.0000


. 
